# 응용 04 · 제공 데이터로 진행한 YOLO 학습 기록

셀은 위에서 아래로 **Shift+Enter**로 실행합니다. 계산·표시에는 pandas가 필요합니다. 이 문서는 제공된 square·circle·triangle 도형 데이터만 다룹니다.

결과 파일 자체는 Git에 포함하지 않습니다. 대신 문서 작성 시 읽은 관측 수치를 코드 셀에 기록해, outputs가 없는 환경에서도 표와 계산을 볼 수 있게 했습니다. 저장된 수치는 과거 실행의 기록이며 문서를 다시 실행해 모델을 학습하는 것은 아닙니다. 실제 모델 호출은 Markdown 코드 예제로 제공합니다. 검증 여부·제출용 판단은 직접 작성하세요.

## 문서의 범위와 근거

이 폴더의 `04_train.py`와 제공된 `data/`를 따라 학습 준비 → 설정 → 학습 기록 → 결과 파일 순서로 설명합니다. 설정값은 `outputs/cli_smoke/args.yaml`, 수치는 `outputs/cli_smoke/results.csv`에 근거합니다. 모델은 이 실습 코드에 지정된 `yolo26n.yaml`만 다룹니다.

## 1. 이번 실행의 입력과 목표

객체 탐지는 이미지 안의 물체마다 종류와 위치를 찾는 작업입니다. train 이미지는 모델 가중치를 바꾸는 데 사용하고 val 이미지는 학습 중 상태를 평가하는 데 사용합니다. test는 이번 학습·평가·confidence 선택에 전달하지 않습니다.

|분할|이미지|라벨 객체|빈 라벨 이미지|
|---|---:|---:|---:|
|train|36|90|6|
|val|12|30|2|
|test|12|30|2|

객체가 있는 이미지에는 세 종류의 도형이 각각 하나씩 있습니다. 빈 라벨 파일은 해당 이미지에 탐지 대상이 없다는 표현입니다. 이미지 수와 객체 수는 다른 단위입니다.

이번 설정은 사전학습 없이 2 epoch를 수행하는 짧은 학습입니다. 모델은 `yolo26n.yaml` 구조에서 시작하고, 5·6번은 여기서 생성한 `last.pt`를 사용합니다.

In [1]:
from pathlib import Path
import pandas as pd

roots = [Path.cwd(), *Path.cwd().parents]
BASE = next((root / "20261106/02_advanced" for root in roots
             if (root / "20261106/02_advanced/04_train.py").is_file()), None)
if BASE is None:
    raise FileNotFoundError("저장소 안에서 문서를 실행하세요.")
print("실습 폴더:", BASE)


실습 폴더: /home/swchoi/projects/python/practice_assignment/20261106/02_advanced


## 2. 파일과 Python 문법

`BASE = Path(__file__).resolve().parent`는 실행 파일이 있는 폴더를 찾습니다. `__file__`은 일반 Python 파일에서 쓰는 변수라 Jupyter 셀에서는 위처럼 저장소 위치를 탐색합니다.

`if __name__ == "__main__":` 안의 코드는 해당 파일을 직접 실행할 때 동작합니다. import만 해도 학습이 시작되는 일을 막고 Windows의 작업자 프로세스 실행에도 대비합니다.

`yaml.safe_dump(config)`는 데이터 경로·클래스 설정 딕셔너리를 YAML 문자열로 바꾸며 `write_text()`로 저장합니다. `YOLO("yolo26n.yaml")`은 구조 정의로 모델을 만들고, 사전학습 `.pt`를 읽는 것과 출발점이 다릅니다.

In [2]:
config = {"train": str(BASE / "data/images/train"),
          "val": str(BASE / "data/images/val"),
          "names": ["square", "circle", "triangle"]}
print(config)


{'train': '/home/swchoi/projects/python/practice_assignment/20261106/02_advanced/data/images/train', 'val': '/home/swchoi/projects/python/practice_assignment/20261106/02_advanced/data/images/val', 'names': ['square', 'circle', 'triangle']}


## 3. 학습 인자를 하나씩 읽기

|인자|기록값|의미|
|---|---|---|
|epochs|2|train 전체를 보는 반복 횟수|
|batch|4|한 번의 학습 계산에 묶는 이미지 수|
|imgsz|160|모델 입력 크기 설정|
|optimizer|SGD|가중치를 갱신하는 방법|
|lr0|0.01|초기 학습률|
|seed|42|난수 시작값|
|workers|0|별도 데이터 로딩 작업자 수|
|amp|False|혼합 정밀도 비활성화|
|pretrained|False|사전학습 가중치 사용하지 않음|
|plots|False|학습 그래프 생성 설정|
|project / name|outputs / cli_smoke|결과 저장 위치|
|exist_ok|True|같은 실행 이름의 폴더를 재사용|

일반적인 이미지 batch 수를 계산하면 36/4=9입니다. 다만 optimizer 갱신 횟수는 gradient accumulation 같은 내부 설정에도 영향을 받으므로 batch 수와 항상 같다고 단정하지 않습니다.

`device=0 if torch.cuda.is_available() else "cpu"`는 GPU를 사용할 수 있으면 첫 번째 GPU를 고르고, 아니면 CPU를 사용합니다. 숫자 1은 CPU가 아니라 두 번째 GPU의 번호입니다. 현재 코드는 CPU 대체값과 API·CLI 장치 기록, 출력 안내 경로를 맞췄습니다.

In [3]:
settings = {"model": "yolo26n.yaml", "epochs": 2, "batch": 4, "imgsz": 160,
            "optimizer": "SGD", "lr0": 0.01, "seed": 42,
            "workers": 0, "amp": False, "pretrained": False,
            "name": "cli_smoke", "recorded_training_device": "0"}
print(pd.Series(settings).to_string())
print("epoch당 이미지 배치 수:", (36 + settings["batch"] - 1) // settings["batch"])


model                       yolo26n.yaml
epochs                                 2
batch                                  4
imgsz                                160
optimizer                            SGD
lr0                                 0.01
seed                                  42
workers                                0
amp                                False
pretrained                         False
name                           cli_smoke
recorded_training_device               0
epoch당 이미지 배치 수: 9


## 4. 실제 학습 기록 읽기

관측값 출처는 `outputs/cli_smoke/results.csv`, 설정 출처는 같은 폴더의 `args.yaml`입니다. args에는 `device: "0"`이 기록돼 있습니다. 아래 표는 기존 GPU 학습 로그를 읽은 기록이며 이번 문서 작성에서 학습을 다시 실행하지 않았습니다.

`train/box_loss`는 위치 관련 학습 오차, `train/cls_loss`는 클래스 관련 학습 오차, `train/l1_loss`는 YOLO26에서 기록한 L1 관련 항목입니다. loss는 학습용 오차이고 mAP처럼 맞힌 비율을 직접 뜻하지 않습니다. 손실 구성은 모델 버전에 따라 다르며, 세 항목을 단순히 더해 성능 점수로 해석하지 않습니다.

`val/…`은 검증 데이터에서 계산한 손실입니다. `metrics/…`의 precision·recall·mAP는 검증 데이터의 탐지 지표입니다.

In [4]:
train_records = [{'epoch': 1.0, 'time': 2.87885, 'train/box_loss': 3.1317, 'train/cls_loss': 4.30248, 'train/l1_loss': 0.04265, 'metrics/precision(B)': 0.0, 'metrics/recall(B)': 0.0, 'metrics/mAP50(B)': 0.0, 'metrics/mAP50-95(B)': 0.0, 'val/box_loss': 2.01265, 'val/cls_loss': 4.4482, 'val/l1_loss': 0.02117, 'lr/pg0': 0.00888889, 'lr/pg1': 0.00888889, 'lr/pg2': 0.02}, {'epoch': 2.0, 'time': 4.04341, 'train/box_loss': 3.1963, 'train/cls_loss': 4.36703, 'train/l1_loss': 0.04172, 'metrics/precision(B)': 0.0, 'metrics/recall(B)': 0.0, 'metrics/mAP50(B)': 0.0, 'metrics/mAP50-95(B)': 0.0, 'val/box_loss': 2.02727, 'val/cls_loss': 4.45075, 'val/l1_loss': 0.02142, 'lr/pg0': 0.00505, 'lr/pg1': 0.00505, 'lr/pg2': 0.00505}]
train_table = pd.DataFrame(train_records)
columns = ["epoch", "time", "train/box_loss", "train/cls_loss", "train/l1_loss",
           "val/box_loss", "val/cls_loss", "val/l1_loss",
           "metrics/precision(B)", "metrics/recall(B)",
           "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
print(train_table[columns].to_string(index=False))


 epoch    time  train/box_loss  train/cls_loss  train/l1_loss  val/box_loss  val/cls_loss  val/l1_loss  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  metrics/mAP50-95(B)
   1.0 2.87885          3.1317         4.30248        0.04265       2.01265       4.44820      0.02117                   0.0                0.0               0.0                  0.0
   2.0 4.04341          3.1963         4.36703        0.04172       2.02727       4.45075      0.02142                   0.0                0.0               0.0                  0.0


## 5. 1 epoch와 2 epoch 사이의 수치 변화

아래 계산은 2행 값에서 1행 값을 뺍니다. 음수이면 감소, 양수이면 증가입니다. `iloc[0]`과 `iloc[-1]`은 각각 첫 행과 마지막 행입니다. 두 epoch만으로 장기 추세나 원인을 확정하지 않습니다.

In [5]:
loss_columns = ["train/box_loss", "train/cls_loss", "train/l1_loss",
                "val/box_loss", "val/cls_loss", "val/l1_loss"]
change = pd.DataFrame({"epoch1": train_table.iloc[0][loss_columns],
                       "epoch2": train_table.iloc[-1][loss_columns]})
change["차이(epoch2-epoch1)"] = change["epoch2"] - change["epoch1"]
print(change.to_string())
print("마지막 행의 time:", train_table.iloc[-1]["time"], "초")


                 epoch1   epoch2  차이(epoch2-epoch1)
train/box_loss  3.13170  3.19630            0.06460
train/cls_loss  4.30248  4.36703            0.06455
train/l1_loss   0.04265  0.04172           -0.00093
val/box_loss    2.01265  2.02727            0.01462
val/cls_loss    4.44820  4.45075            0.00255
val/l1_loss     0.02117  0.02142            0.00025
마지막 행의 time: 4.04341 초


두 epoch 모두 precision·recall·mAP50·mAP50–95가 0으로 기록됐습니다. box·cls 학습 손실은 각각 3.1317→3.1963, 4.30248→4.36703이며 L1 학습 손실은 0.04265→0.04172입니다. `time`의 마지막 값 4.04341초는 CSV에 기록된 누적 학습 시간 항목이며, 프로그램 시작부터 모델 구성·저장까지 포함한 전체 벽시계 시간과 구분합니다.

**관측:** 위 표에 기록된 지표와 손실 변화입니다. **원인 가설:** 무작위 초기화와 짧은 학습 기간이 결과에 영향을 줬을 수 있습니다. epoch·사전학습 여부를 바꾼 비교 실행을 하지 않았으므로 원인으로 확정하지 않습니다.

## 6. API·CLI·저장 파일의 연결

Python API의 핵심 호출은 다음과 같습니다. 실제 학습을 다시 시작하므로 필요할 때만 실행합니다.

```python
model.train(data=path, epochs=2, imgsz=160, batch=4, workers=0,
            device=device, amp=False, pretrained=False,
            optimizer="SGD", lr0=0.01, seed=42, plots=False,
            project=out, name="cli_smoke", exist_ok=True)
```

CLI는 `yolo detect train model=yolo26n.yaml data="..." epochs=2 ...` 형태입니다. 원본 파일은 API와 같은 device를 포함한 명령 문자열을 `same_settings_cli.txt`에 저장합니다. 문자열을 저장하는 동작과 CLI를 실제 실행하는 동작은 다릅니다.

|파일|역할|
|---|---|
|dataset_active.yaml|이번 실행의 데이터 경로·클래스 설정|
|cli_smoke/args.yaml|학습 설정 기록|
|cli_smoke/results.csv|epoch별 손실·평가 수치|
|cli_smoke/weights/best.pt|선택 기준에 따른 체크포인트|
|cli_smoke/weights/last.pt|마지막 epoch 체크포인트|

5·6번은 **last.pt**를 선택합니다. best와 last가 있다는 사실만으로 두 모델의 성능 차이를 알 수는 없습니다. 가중치·CSV·캐시·실행 폴더는 커밋에 추가하지 않습니다.

## 7. 직접 기록

|항목|직접 작성|
|---|---|
|실행·검증 여부와 근거||
|고정한 조건과 바꾼 조건||
|관측한 한계와 다음 비교 계획||

참고한 로컬 파일: `04_train.py`, `data/manifest.csv`, `data/labels/`, `outputs/cli_smoke/args.yaml`, `outputs/cli_smoke/results.csv`.